# Semana 04: Atividade de Pesquisa em Duplas — Arquitetura IoT na AWS (Mobile + MQTT + Node-RED)

## Revisão Integrada e Aplicação Prática: IaaS, PaaS, EC2, Docker e Redes na AWS

Nesta semana consolidamos os conhecimentos desenvolvidos no primeiro bloco da disciplina (**Semanas 01 a 03**).
Em duplas, vocês atuarão como **Arquitetos de Soluções em Nuvem** para desenhar e justificar a infraestrutura na **AWS** de uma solução real de monitoramento industrial da **Fábrica Virtual Smart N1**.

### Contexto do Desafio
Vocês foram encarregados de desenhar a infraestrutura em nuvem na **AWS** para uma solução completa de monitoramento industrial. O sistema consiste em:
1. **Sensores industriais / CLPs no chão de fábrica:** coletando dados de telemetria e enviando mensagens via protocolo **MQTT**.
2. **Node-RED na nuvem:** processando os dados, aplicando regras de negócio, gerando alertas e alimentando dashboards.
3. **Aplicativo Mobile / Dashboard Web:** permitindo que engenheiros e operadores acompanhem o status das máquinas e recebam alertas em tempo real.

---


## 1. Síntese dos Conceitos Fundamentais (Semanas 01 a 03)

Utilizem a matriz de conceitos abaixo para embasar as escolhas técnicas da proposta:

| Conceito / Serviço | Categoria | Papel na Arquitetura | Visto em |
| :--- | :--- | :--- | :--- |
| **IaaS vs PaaS** | Modelo de Serviço | Nível de responsabilidade e controle sobre a infraestrutura x esforço de gerência | Semana 01 |
| **Regiões AWS** | Infraestrutura Global | Escolha entre `us-east-1` (N. Virgínia - menor custo) e `sa-east-1` (São Paulo - menor latência/LGPD) | Semana 02 |
| **Amazon EC2** | IaaS / Computação | Máquinas virtuais Linux (ex: `t3.micro`/`t2.micro`) para hospedar aplicações e containers | Semana 03 |
| **Docker Engine** | Conteinerização | Isolamento, portabilidade e agilidade no deploy do Node-RED e brokers | Semana 03 |
| **Docker Volumes** | Armazenamento | Persistência do diretório de dados (`/data`) fora do ciclo de vida efêmero do container | Semana 03 |
| **Security Groups** | Segurança e Rede | Firewall virtual com regras de entrada (*Inbound*) controlando portas (SSH, HTTP, MQTT, UI) | Semana 03 |

---


## 2. Atividade Prática de Pesquisa — O que a dupla deve entregar

A dupla deve produzir um documento técnico (de 1 a 2 páginas, em **PDF ou Markdown**) respondendo de forma detalhada e fundamentada aos **4 tópicos** a seguir:

---

### Tópico 1: Hospedagem do Node-RED (IaaS vs. Containers)

Comparando o que estudamos nas aulas sobre máquinas virtuais (EC2) e conteinerização (Docker):
1. **Como vocês sugerem hospedar o Node-RED na AWS?**
2. **Qual tipo de instância EC2** (ex: `t3.micro`, `t3.small`) e qual **sistema operacional (AMI)** utilizariam? Justifiquem com base em custo e requisitos de recursos.
3. **Como garantir que os fluxos do Node-RED não se percam** caso o container seja reiniciado, atualizado ou recriado? Explique o uso de **Docker Volumes** mapeando o diretório interno `/data` para o armazenamento persistente (EBS) da EC2.

---


### Tópico 2: Broker MQTT na AWS (IaaS vs. PaaS)

Pesquisem e comparem as duas opções viáveis para gerenciar as mensagens MQTT na nuvem:

| Critério de Comparação | Opção A: IaaS (Eclipse Mosquitto na EC2) | Opção B: PaaS (AWS IoT Core Nativo) |
| :--- | :--- | :--- |
| **Tipo de Arquitetura** | Container próprio gerenciado na máquina virtual | Serviço nativo totalmente gerenciado pela AWS |
| **Gerenciamento** | Você cuida de SO, patches, alta disponibilidade e escalabilidade | A AWS gerencia escala, segurança e disponibilidade |
| **Modelo de Cobrança** | Custo fixo por hora da instância EC2 | Cobrança por volume de mensagens e conexões ativas |
| **Segurança e Certificados** | Configuração manual de certificados SSL/TLS no Mosquitto | Gerenciamento nativo de certificados X.509 por dispositivo |

**Pergunta para a dupla:**  
Qual das duas opções vocês escolheriam para a solução da fábrica? Justifiquem levando em consideração o **custo**, a **manutenção operacional** e o **Modelo de Responsabilidade Compartilhada da AWS**.

---


### Tópico 3: Segurança e Rede (Regiões AWS e Security Groups)

1. **Escolha da Região AWS:**
   - Em qual região a infraestrutura será provisionada: `sa-east-1` (São Paulo) ou `us-east-1` (N. Virgínia)?
   - Analisem a relação de compromisso (*trade-off*) entre **latência para o chão de fábrica no Brasil**, **custo em dólares dos serviços** e conformidade com a **LGPD**.

2. **Configuração de Firewall (Security Group):**
   - Complete a tabela de regras de entrada (*Inbound Rules*) necessárias para a solução operar com segurança:

| Porta | Protocolo | Serviço / Finalidade | Origem Recomendada (*Source*) | Justificativa de Segurança |
| :---: | :---: | :--- | :--- | :--- |
| **22** | TCP | SSH (Acesso ao terminal Linux da EC2) | `Meu IP` / IP da VPN da fábrica | Nunca deixar aberto para `0.0.0.0/0` |
| **80** | TCP | HTTP (Dashboard web público ou Proxy Nginx) | `0.0.0.0/0` ou Rede Local | Acesso web dos operadores |
| **1880** | TCP | Interface do Editor Node-RED | IP do Administrador / VPN | Proteger o editor de fluxos contra acessos indevidos |
| **1883** | TCP | MQTT Padrão (Sem criptografia) | Rede interna dos CLPs / Sensores | Comunicação direta (apenas em redes privadas) |
| **8883** | TCP | MQTT com Criptografia TLS | Sensores externos / Dispositivos remotos | Tráfego seguro via Internet |
| **443** | TCP | HTTPS / AWS IoT Core (WebSockets) | `0.0.0.0/0` | Conexões seguras de dashboards e apps mobile |

---


### Tópico 4: Diagrama de Blocos da Solução

Elaborem um diagrama de arquitetura em blocos (utilizando ferramentas como **Excalidraw**, **draw.io**, **Lucidchart** ou desenho claro):

```text
  +----------------------------+             +-----------------------------------------+
  |   CHÃO DE FÁBRICA / IOT    |             |             NUVEM AWS                   |
  |                            |             |                                         |
  | [ Sensores de Temperatura ]|             |  [ BROKER MQTT ]                        |
  |             |              |   (MQTT)    |  (Mosquitto na EC2 ou AWS IoT Core)     |
  |             v              | ----------> |                     |                   |
  | [ CLPs Industriais / Edge ]|  Porta 8883 |                     v                   |
  +----------------------------+             |  [ NODE-RED CONTAINER ]                 |
                                             |  (EC2 Ubuntu + Docker Volume)           |
                                             |                     |                   |
  +----------------------------+             |                     v (HTTPS / WS)      |
  |  OPERADORES / SUPERVISÃO   | <---------------------------------+ Porta 443 / 1880  |
  | [ App Mobile / Dashboard ] |             |                                         |
  +----------------------------+             +-----------------------------------------+
```

**Requisitos obrigatórios do diagrama:**
- Identificar claramente o que está no ambiente local (*On-Premises / Fábrica*) e o que está na **AWS**.
- Indicar o tipo de serviço utilizado em cada bloco (ex: `EC2 IaaS`, `AWS IoT Core PaaS`, `Docker Volume`).
- Indicar os **protocolos e portas** em cada fluxo de conexão.
- Indicar a **Região AWS** escolhida.

---


## 3. Guia de Referência Rápida para o Aluno

Abaixo estão os comandos essenciais de Docker e AWS estudados nas aulas práticas para consulta durante a elaboração do trabalho:

```bash
# 1. Criar volume Docker nomeado para persistência do Node-RED
docker volume create nodered_data

# 2. Executar o container do Node-RED com reinício automático e volume mapeado
docker run -d \
  --name nodered-smartn1 \
  --restart always \
  -p 1880:1880 \
  -v nodered_data:/data \
  nodered/node-red:latest

# 3. Inspecionar o status do container e logs de inicialização
docker ps
docker logs --tail 30 nodered-smartn1

# 4. Acesso ao editor de fluxos no navegador
# http://IP_PUBLICO_DA_EC2:1880
```

---


## 4. Instruções e Critérios de Avaliação

- **Formato:** Documento em PDF ou Markdown com os nomes completos da dupla.
- **Tamanho recomendado:** Entre 1 e 2 páginas de texto explicativo + diagrama de arquitetura.
- **Critérios de Avaliação:**
  1. **Coerência Técnica:** Escolha adequada de tipos de instância, AMIs, portas e regiões com justificativas reais.
  2. **Domínio de Conceitos:** Uso correto dos termos e conceitos de IaaS, PaaS, Docker Volumes e Security Groups.
  3. **Qualidade do Diagrama:** Clareza no fluxo de dados, separação de ambientes e indicação de portas/protocolos.
  4. **Segurança:** Adoção de boas práticas no firewall (restringir portas administrativas e privilegiar criptografia).
